# Stakeholder demonstration: sentiment and topic classification

This executable notebook trains both LSTM classifiers and presents their results in a stakeholder-oriented dashboard. All reusable data, model, prediction, and visualization functions live in `src/`.

## Goal

Demonstrate how product reviews can be classified by sentiment and product topic, show model performance and failure modes, and translate predictions into an example routing decision.

## Setup

In [1]:
from pathlib import Path
import os
import sys

from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run the notebook from inside the repository.')

sys.path.insert(0, str(ROOT))

from src.sentiment_classifier import SentimentPipelineConfig, run_sentiment_pipeline
from src.topic_classifier import TopicPipelineConfig, run_topic_pipeline
from src.text_classifier.visualization import (
    render_stakeholder_dashboard,
    write_stakeholder_overview,
)

print('Environment configured.')

Environment configured.


### Key assumptions

- Both datasets are synthetic, balanced, and intentionally small.
- Results demonstrate pipeline behavior, not production readiness.
- Real deployment requires representative data, confidence calibration, monitoring, and business acceptance thresholds.

## Steps

### 1. Configure both classifiers and stakeholder examples

Edit `demo_reviews` to classify different reviews. The same text is sent to both models.

In [2]:
epochs = int(os.getenv('PIPELINE_EPOCHS', '20'))

demo_reviews = (
    'The smartphone battery is terrible and drains too quickly',
    'The television works perfectly and the picture is excellent',
    'The refrigerator arrived on Monday in a standard package',
    'The washing machine is unreliable and stops during the cycle',
)

sentiment_config = SentimentPipelineConfig(
    dataset_path=ROOT / 'data' / 'sentiment_samples.csv',
    epochs=epochs,
    seed=42,
    demo_texts=demo_reviews,
)
topic_config = TopicPipelineConfig(
    dataset_path=ROOT / 'data' / 'topic_samples.csv',
    epochs=epochs,
    seed=42,
    demo_texts=demo_reviews,
)

### 2. Train and evaluate sentiment and topic models

In [3]:
sentiment_result = run_sentiment_pipeline(sentiment_config)
topic_result = run_topic_pipeline(topic_config)

print('Both classifiers trained and evaluated.')

Both classifiers trained and evaluated.


In [4]:
model_summary = {
    'sentiment': {
        'dataset_size': sentiment_result.dataset_size,
        'train_size': sentiment_result.train_size,
        'test_size': sentiment_result.test_size,
        'labels': sentiment_result.labels,
        'test_accuracy': round(sentiment_result.metrics['accuracy'], 4),
        'test_loss': round(sentiment_result.metrics['loss'], 4),
    },
    'topic': {
        'dataset_size': topic_result.dataset_size,
        'train_size': topic_result.train_size,
        'test_size': topic_result.test_size,
        'labels': topic_result.labels,
        'test_accuracy': round(topic_result.metrics['accuracy'], 4),
        'test_loss': round(topic_result.metrics['loss'], 4),
    },
    'epochs': epochs,
}

model_summary

{'sentiment': {'dataset_size': 90,
  'train_size': 72,
  'test_size': 18,
  'labels': ['negative', 'neutral', 'positive'],
  'test_accuracy': 1.0,
  'test_loss': 0.0258},
 'topic': {'dataset_size': 80,
  'train_size': 64,
  'test_size': 16,
  'labels': ['refrigerator', 'smartphone', 'television', 'washing_machine'],
  'test_accuracy': 0.6875,
  'test_loss': 0.9746},
 'epochs': 20}

### 3. Render stakeholder visualizations

The dashboard shows executive metrics, learning curves, class balance, confusion matrices, prediction review, and a dual-classifier business demonstration with confidence profiles.

In [5]:
stakeholder_dashboard = render_stakeholder_dashboard(
    sentiment_result,
    topic_result,
)
display(HTML(stakeholder_dashboard))

In [6]:
overview_path = write_stakeholder_overview(
    ROOT / 'docs' / 'assets' / 'stakeholder_overview.svg',
    sentiment_result,
    topic_result,
)
print('README overview generated:', overview_path.relative_to(ROOT))

README overview generated: docs\assets\stakeholder_overview.svg


## Checks

In [7]:
assert set(sentiment_result.labels) == {'negative', 'neutral', 'positive'}
assert set(topic_result.labels) == {
    'smartphone', 'television', 'refrigerator', 'washing_machine'
}
assert sentiment_result.metrics['accuracy'] >= 0.50, sentiment_result.metrics
assert topic_result.metrics['accuracy'] >= 0.50, topic_result.metrics
assert len(sentiment_result.predictions) == sentiment_result.test_size
assert len(topic_result.predictions) == topic_result.test_size
assert len(sentiment_result.demo_predictions) == len(demo_reviews)
assert len(topic_result.demo_predictions) == len(demo_reviews)
assert all(0.0 <= item['confidence'] <= 1.0 for item in sentiment_result.demo_predictions)
assert all(0.0 <= item['confidence'] <= 1.0 for item in topic_result.demo_predictions)
assert 'Business demonstration' in stakeholder_dashboard
assert overview_path.is_file()

{
    'status': 'ok',
    'sentiment_accuracy': round(sentiment_result.metrics['accuracy'], 4),
    'topic_accuracy': round(topic_result.metrics['accuracy'], 4),
    'demo_reviews': len(demo_reviews),
    'stakeholder_dashboard': True,
}

{'status': 'ok',
 'sentiment_accuracy': 1.0,
 'topic_accuracy': 0.6875,
 'demo_reviews': 4,
 'stakeholder_dashboard': True}

## Next Steps

Replace synthetic data with representative, labeled reviews; agree on acceptance thresholds with business owners; calibrate confidence; and monitor class-level quality and drift before production use.